# Continuous Distributions
**Topic:** Random Variables & Distributions

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** why probability for a continuous variable is always an area, never a point
- **Describe** the Uniform and Exponential distributions and their real-world meaning
- **Interpret** a probability density function (PDF) and calculate probabilities using the area under the curve

> **Tip:** Start by examining the **Uniform distribution**, drag the bounds and notice that the height of the rectangle adjusts so the total area always stays exactly 1.

---
## How we got here

In *09–10* we worked with discrete distributions where we could list every possible outcome and assign it a probability. Continuous variables, height, time, temperature, can take any real value, so we cannot assign probability to a single point. This notebook introduces the probability density function, which replaces the PMF for continuous variables and opens the door to the family of continuous distributions.

---
## Why this matters for data science

Continuous distributions model the measurements that flow through every ML pipeline: input features like age, income, and duration; model outputs like predicted probabilities; and errors like residuals. The assumption that residuals follow a normal distribution underlies linear regression; the assumption that inter-event times follow an exponential distribution underlies survival analysis and queueing models used in infrastructure planning.

---
## Try it yourself

In [2]:
from plotly.subplots import make_subplots

out         = Output()
summary_out = Output()

_STYLE  = {"description_width": "110px"}
_LAYOUT = widgets.Layout(width="440px")

dist_dd  = Dropdown(
    options=["Uniform", "Exponential", "Beta"],
    value="Uniform",
    description="Distribution:",
    style=_STYLE,
    layout=_LAYOUT,
)

a_sl     = FloatSlider(value=-2.0, min=-5.0, max=4.0,  step=0.25, description="a (lower):", style=_STYLE, layout=_LAYOUT)
b_sl     = FloatSlider(value=2.0,  min=-4.0, max=5.0,  step=0.25, description="b (upper):", style=_STYLE, layout=_LAYOUT)
lam_sl   = FloatSlider(value=1.0,  min=0.25, max=5.0,  step=0.25, description="\u03bb (rate):",  style=_STYLE, layout=_LAYOUT)
alpha_sl = FloatSlider(value=2.0,  min=0.5,  max=10.0, step=0.5,  description="\u03b1 (alpha):", style=_STYLE, layout=_LAYOUT)
beta_sl  = FloatSlider(value=2.0,  min=0.5,  max=10.0, step=0.5,  description="\u03b2 (beta):",  style=_STYLE, layout=_LAYOUT)

range_lo_sl = FloatSlider(value=-1.0, min=-2.0, max=2.0, step=0.05, description="Range a:", style=_STYLE, layout=_LAYOUT)
range_hi_sl = FloatSlider(value=1.0,  min=-2.0, max=2.0, step=0.05, description="Range b:", style=_STYLE, layout=_LAYOUT)

uniform_box = VBox([a_sl, b_sl])
exp_box     = VBox([lam_sl])
beta_box    = VBox([alpha_sl, beta_sl])
exp_box.layout.display  = "none"
beta_box.layout.display = "none"

_busy = [False]

def _support():
    d = dist_dd.value
    if d == "Uniform":
        a, b = a_sl.value, b_sl.value
        return (a, b) if b > a + 0.1 else (a, a + 0.25)
    if d == "Exponential":
        sc = 1.0 / lam_sl.value
        return (0.0, round(6.0 * sc, 2))
    return (0.0, 1.0)

def _dist_obj():
    d = dist_dd.value
    if d == "Uniform":
        a = a_sl.value
        b = max(b_sl.value, a + 0.01)
        return stats.uniform(loc=a, scale=b - a)
    if d == "Exponential":
        return stats.expon(scale=1.0 / lam_sl.value)
    return stats.beta(alpha_sl.value, beta_sl.value)

def _safe_set(sl, lo, hi, val, step):
    val = float(np.clip(val, lo, hi))
    sl.step = step
    if val > sl.max:
        sl.max = hi; sl.value = val; sl.min = lo
    elif val < sl.min:
        sl.min = lo; sl.value = val; sl.max = hi
    else:
        sl.min = lo; sl.max = hi; sl.value = val

def _sync_range(keep):
    lo, hi = _support()
    step = 0.01 if dist_dd.value == "Beta" else 0.05
    _busy[0] = True
    _safe_set(range_lo_sl, lo, hi, range_lo_sl.value if keep else lo, step)
    _safe_set(range_hi_sl, lo, hi, range_hi_sl.value if keep else hi, step)
    _busy[0] = False

def render():
    d_obj = _dist_obj()
    lo, hi = _support()
    dist = dist_dd.value

    rlo = float(np.clip(range_lo_sl.value, lo, hi))
    rhi = float(np.clip(range_hi_sl.value, lo, hi))
    if rhi <= rlo:
        rhi = min(rlo + 1e-3, hi)

    pad = (hi - lo) * 0.08
    if dist == "Exponential":
        x = np.linspace(0.0, hi + pad, 700)
    elif dist == "Beta":
        x = np.linspace(0.005, 0.995, 700)
    else:
        x = np.linspace(lo - pad, hi + pad, 700)

    pdf_v = d_obj.pdf(x)
    cdf_v = d_obj.cdf(x)

    mask  = (x >= rlo) & (x <= rhi)
    x_sh  = x[mask]
    pdf_sh = pdf_v[mask]

    prob    = float(d_obj.cdf(rhi) - d_obj.cdf(rlo))
    cdf_rhi = float(d_obj.cdf(rhi))
    mean_v  = float(d_obj.mean())
    var_v   = float(d_obj.var())

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Probability Density  (PDF)", "Cumulative Distribution  (CDF)"),
        horizontal_spacing=0.13,
    )

    fig.add_trace(go.Scatter(
        x=x, y=pdf_v, mode="lines",
        line=dict(color=PALETTE["primary"], width=3),
        name="PDF", showlegend=False,
    ), row=1, col=1)

    if len(x_sh) >= 2:
        fig.add_trace(go.Scatter(
            x=np.concatenate([[rlo], x_sh, [rhi]]),
            y=np.concatenate([[0],   pdf_sh, [0]]),
            fill="toself", fillcolor="rgba(76,110,245,0.22)",
            line=dict(color=PALETTE["primary"], width=0),
            name=f"P = {prob:.4f}", showlegend=True,
        ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=x, y=cdf_v, mode="lines",
        line=dict(color=PALETTE["secondary"], width=3),
        name="CDF", showlegend=False,
    ), row=1, col=2)

    fig.add_shape(type="line",
        x0=rhi, x1=rhi, y0=0, y1=cdf_rhi,
        line=dict(color=PALETTE["secondary"], width=1.5, dash="dash"),
        row=1, col=2)
    fig.add_shape(type="line",
        x0=x[0], x1=rhi, y0=cdf_rhi, y1=cdf_rhi,
        line=dict(color=PALETTE["secondary"], width=1.5, dash="dash"),
        row=1, col=2)

    bl = base_layout(title=f"{dist} Distribution", xaxis_title="x", yaxis_title="Density")
    bl.update(dict(height=430, showlegend=True, legend=dict(orientation="h", y=-0.14)))
    fig.update_layout(bl)
    fig.update_xaxes(title_text="x")
    fig.update_yaxes(title_text="Density", row=1, col=1)
    fig.update_yaxes(title_text="Cumulative Probability", range=[-0.02, 1.06], row=1, col=2)

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    if dist == "Uniform":
        a, b  = a_sl.value, b_sl.value
        param_s = f"a = {a:.2f}, b = {b:.2f}"
        formula = (f"stats.uniform.cdf({rhi:.2f}, loc={a:.2f}, scale={b-a:.2f})"
                   f" \u2212 stats.uniform.cdf({rlo:.2f}, loc={a:.2f}, scale={b-a:.2f})")
    elif dist == "Exponential":
        sc = 1.0 / lam_sl.value
        param_s = f"\u03bb = {lam_sl.value:.2f}"
        formula = (f"stats.expon.cdf({rhi:.2f}, scale={sc:.4f})"
                   f" \u2212 stats.expon.cdf({rlo:.2f}, scale={sc:.4f})")
    else:
        a, b  = alpha_sl.value, beta_sl.value
        param_s = f"\u03b1 = {a:.1f}, \u03b2 = {b:.1f}"
        formula = (f"stats.beta.cdf({rhi:.2f}, {a:.1f}, {b:.1f})"
                   f" \u2212 stats.beta.cdf({rlo:.2f}, {a:.1f}, {b:.1f})")

    html_s = (
        f'<div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;'
        f'padding:14px 22px;margin-top:6px;line-height:2.0;font-size:13px;">'
        f'<strong style="font-size:14px;">{dist} Distribution</strong>'
        f'&nbsp;&mdash;&nbsp;{param_s}<br>'
        f'P({rlo:.2f} &lt; X &lt; {rhi:.2f})'
        f'&nbsp;=&nbsp;<strong>{prob:.4f}</strong>'
        f'&nbsp;=&nbsp;<strong>{prob * 100:.2f}%</strong><br>'
        f'<code style="color:#495057;">{formula}</code><br>'
        f'E[X] = {mean_v:.4f}&nbsp;&nbsp;&nbsp;Var(X) = {var_v:.4f}'
        f'</div>'
    )

    with summary_out:
        clear_output(wait=True)
        display(HTML(html_s))

def on_dist(change):
    d = dist_dd.value
    uniform_box.layout.display = "" if d == "Uniform"     else "none"
    exp_box.layout.display     = "" if d == "Exponential" else "none"
    beta_box.layout.display    = "" if d == "Beta"        else "none"
    _sync_range(keep=False)
    render()

def on_param(change):
    _sync_range(keep=True)
    render()

def on_range(change):
    if not _busy[0]:
        render()

dist_dd.observe(on_dist, names="value")
for _sl in [a_sl, b_sl, lam_sl, alpha_sl, beta_sl]:
    _sl.observe(on_param, names="value")
range_lo_sl.observe(on_range, names="value")
range_hi_sl.observe(on_range, names="value")

controls     = VBox([dist_dd, uniform_box, exp_box, beta_box, range_lo_sl, range_hi_sl])
_widget_box  = VBox([controls, out, summary_out])

if not hasattr(out, "_initialized"):
    display(_widget_box)
    out._initialized = True

render()


---
## What's happening?

For a continuous random variable, **probability is area**, not height. The height of the probability density function (PDF) tells you the *relative likelihood* of values nearby, but the actual probability of any single point is exactly zero.

| Distribution | Parameters | Shape | Use case |
|-------------|-----------|-------|---------|
| Uniform(a, b) | a = min, b = max | Flat rectangle | All outcomes equally likely in a range |
| Exponential(λ) | λ = rate | Right-skewed decay | Time between independent events |
| Beta(α, β) | α, β > 0 | Flexible (0 to 1) | Modeling probabilities and proportions |

$$f_{\text{Uniform}}(x) = \frac{1}{b-a}, \quad a \le x \le b$$

$$f_{\text{Exp}}(x) = \lambda e^{-\lambda x}, \quad x \ge 0, \quad E[X] = \frac{1}{\lambda}$$

### The CDF connects area to probability

$$P(a < X < b) = \int_a^b f(x)\,dx = F(b) - F(a)$$

where F is the **cumulative distribution function (CDF)**. In the widget, every shaded region is computing this integral. The key rule: **the total area under any PDF always equals 1**.

---
## Real-world example: Wait times between support tickets

A software company's support team tracks the time between ticket submissions. When tickets arrive independently at a constant average rate, the waiting time between arrivals follows an Exponential distribution.

The chart shows an Exponential PDF calibrated to a support queue that receives roughly 10 tickets per hour (λ = 10/60 ≈ 0.167 per minute, mean wait ≈ 6 minutes). Notice:

- **Notice:** The most probable waiting time is near zero, brief gaps between tickets are more common than long ones
- **Notice:** The distribution has no upper bound; very long gaps are possible but exponentially unlikely
- **Notice:** The shaded area shows P(wait > 10 minutes), a concrete probability readable directly as area under the curve

> **Discussion question:** If the support team wants 90% of ticket gaps to be handled by a single agent before the next one arrives (i.e., they need the 90th percentile of wait time), how would you calculate that from the Exponential distribution?

In [3]:
# ── Exponential distribution: wait time between support tickets ─────────────────
# ~10 tickets/hour → rate = 10/60 per minute
lam   = 10 / 60
mu    = 1 / lam   # mean wait time in minutes

x_max = 30
x_cont = np.linspace(0, x_max, 500)
pdf   = stats.expon.pdf(x_cont, scale=mu)

threshold = 10  # minutes — shade P(X > threshold)
x_shade = x_cont[x_cont >= threshold]
p_shade = stats.expon.sf(threshold, scale=mu)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x_cont, y=pdf,
    mode="lines", line=dict(color=PALETTE["primary"], width=3),
    name=f"Exponential PDF  (λ={lam:.3f}, mean={mu:.1f} min)",
))
fig.add_trace(go.Scatter(
    x=np.concatenate([[threshold], x_shade, [x_shade[-1]]]),
    y=np.concatenate([[0], stats.expon.pdf(x_shade, scale=mu), [0]]),
    fill="toself", fillcolor="rgba(76,110,245,0.20)",
    line=dict(color=PALETTE["primary"], width=0),
    name=f"P(wait > {threshold} min) = {p_shade:.1%}",
))
fig.add_vline(x=mu, line_dash="dash", line_color=PALETTE["secondary"],
              annotation_text=f"Mean = {mu:.1f} min")
layout = base_layout(
    title="Time Between Support Tickets — Exponential Distribution",
    xaxis_title="Wait Time (minutes)",
    yaxis_title="Probability Density",
)
fig.update_layout(layout)
fig.show()

### The continuous distribution family

| Distribution | Range | Shape | Common use case |
|-------------|-------|-------|----------------|
| Uniform(a, b) | [a, b] | Flat | Random number generation, naive priors |
| Exponential(λ) | [0, ∞) | Decaying | Time between events, service times |
| Normal(μ, σ) | (−∞, ∞) | Bell curve | Heights, errors, CLT limit (next notebooks) |
| Beta(α, β) | [0, 1] | Flexible | Modeling probabilities, click-through rates |
| Gamma(α, β) | [0, ∞) | Right-skewed | Sum of exponential waiting times |

---
## Key takeaway

> **For continuous variables, probability is area under the PDF curve, never height at a point — and the total area always equals 1, no matter which continuous distribution you use.**

---
*Next up: The Normal Distribution — the single most important continuous distribution, and why it appears everywhere*